# Hyperparameter Tuning  

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
pd.set_option("display.max_colwidth", None)

df = pd.read_csv("D:\\Heart_Disease_Project\\data\\heart_disease_feature_selected.csv")
#df = pd.read_csv("D:\\Heart_Disease_Project\\data\\heart_diseasePCA.csv")
#df = pd.read_csv("D:\\Heart_Disease_Project\\data\\heart_disease.csv")

X = df.drop(columns=['target'])  
y = df['target']  
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

file_path = 'D:\\Heart_Disease_Project\\results\\evaluation_metrics_NOT_HyperTuned.csv'
comparison_df = pd.read_csv(file_path, index_col= 0)

print(comparison_df)

                            accuracy  precision    recall  f1_score
Logistic Regression         0.885246   0.885477  0.885246  0.885122
Decision Tree               0.803279   0.809808  0.803279  0.803067
Random Forest               0.836066   0.842949  0.836066  0.835889
Support Vector Machine SVM  0.885246   0.885845  0.885246  0.885308


## Use GridSearchCV & RandomizedSearchCV to optimize model hyperparameters. 

### 1-Logistic Regression 

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from scipy.stats import loguniform
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold


LR = LogisticRegression(random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist_LR = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'C': loguniform(0.01, 100), 
    'solver': ['liblinear', 'saga'], 
    'l1_ratio': np.random.rand(100)
}


RSCV_LR_model = RandomizedSearchCV(
    estimator=LR,
    param_distributions=param_dist_LR,
    n_iter=75,
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

RSCV_LR_model.fit(X_train, y_train)
y_pred_RSCV_LR = RSCV_LR_model.predict(X_test)


print("Best parameters found by RandomizedSearchCV:")
print(RSCV_LR_model.best_params_)
print()
print("Accuracy:")
print(f"{accuracy_score(y_test,y_pred_RSCV_LR) * 100:.2f}%")
print()

Fitting 5 folds for each of 75 candidates, totalling 375 fits
Best parameters found by RandomizedSearchCV:
{'C': np.float64(0.6218656551522515), 'l1_ratio': np.float64(0.01647249555107988), 'penalty': 'l1', 'solver': 'saga'}

Accuracy:
88.52%



d:\Heart_Disease_Project\venv\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
70 fits failed out of a total of 375.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
70 fits failed with the following error:
Traceback (most recent call last):
  File "d:\Heart_Disease_Project\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "d:\Heart_Disease_Project\venv\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Heart_Disease_Project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1218, 

#### Comments:
- Best parameters:
{'C': np.float64(0.2241097161910952), 'l1_ratio': np.float64(0.3577183787389395), 'penalty': 'elasticnet', 'solver': 'saga'}

- Accuracy:
88.52%

- The accuracy is the same as before using Random Search
- Used random search first to search in broad range then will search using grid search to refine the results
- Used Stratified K-Fold ensures that each fold has the same class distribution as the original dataset to avoid imbalancement and bias

In [32]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

GSCV_LR = LogisticRegression()

param_grid_LR = {
    'penalty': ['elasticnet'],
    'solver': ['saga'],
    'C': [0.1, 0.2, 0.22, 0.25, 0.3],
    'l1_ratio': [0.25, 0.3, 0.35, 0.4, 0.45]
}

GSCV_LR_model = GridSearchCV(
    estimator=GSCV_LR,
    param_grid=param_grid_LR,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

GSCV_LR_model.fit(X_train, y_train)
y_pred_GSCV_LR = GSCV_LR_model.predict(X_test)


print("\nFinal best parameters found by GridSearchCV:")
print(GSCV_LR_model.best_params_)

final_LR_model = GSCV_LR_model.best_estimator_


report_LR = classification_report(y_test, y_pred_GSCV_LR, output_dict=True)
accuracy_LR = report_LR['accuracy']
precision_LR = report_LR['weighted avg']['precision']
recall_LR = report_LR['weighted avg']['recall']
f1_score_LR = report_LR['weighted avg']['f1-score']

comparison_df.loc["Logistic Regression Tuned"] = [accuracy_LR, precision_LR, recall_LR, f1_score_LR]

print(f"\nModel Accuracy: {accuracy_LR * 100:.2f}%")
print("\nClassification Report:")
print(pd.DataFrame(report_LR))
print(comparison_df)

Fitting 5 folds for each of 25 candidates, totalling 125 fits

Final best parameters found by GridSearchCV:
{'C': 0.2, 'l1_ratio': 0.25, 'penalty': 'elasticnet', 'solver': 'saga'}

Model Accuracy: 90.16%

Classification Report:
                   0         1  accuracy  macro avg  weighted avg
precision   0.896552   0.90625  0.901639   0.901401      0.901639
recall      0.896552   0.90625  0.901639   0.901401      0.901639
f1-score    0.896552   0.90625  0.901639   0.901401      0.901639
support    29.000000  32.00000  0.901639  61.000000     61.000000
                            accuracy  precision    recall  f1_score
Logistic Regression         0.885246   0.885477  0.885246  0.885122
Decision Tree               0.803279   0.809808  0.803279  0.803067
Random Forest               0.836066   0.842949  0.836066  0.835889
Support Vector Machine SVM  0.885246   0.885845  0.885246  0.885308
Logistic Regression Tuned   0.901639   0.901639  0.901639  0.901639


#### Comments:
- The accuracy increased to 90.16%
- Using GridSearch to refine the Search
- Save the best parameters for Logistic Regression

### 2- Decision Tree 

In [33]:
from sklearn.tree import DecisionTreeClassifier

DT = DecisionTreeClassifier(random_state=42)

param_dist_DT = {
   'criterion': ['gini', 'entropy'],
    'max_depth': [5, 10, 15, 20, 25, 30, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', None]
}

RSCV_DT_model = RandomizedSearchCV(
    estimator=DT,
    param_distributions=param_dist_DT,
    n_iter=150,
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

RSCV_DT_model.fit(X_train, y_train)
y_pred_RSCV_DT = RSCV_DT_model.predict(X_test)


print("Best parameters found by RandomizedSearchCV:")
print(RSCV_DT_model.best_params_)
print()
print("Accuracy:")
print(f"{accuracy_score(y_test,y_pred_RSCV_DT)* 100 :.2f}%")
print()

Fitting 5 folds for each of 150 candidates, totalling 750 fits
Best parameters found by RandomizedSearchCV:
{'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 5, 'criterion': 'entropy'}

Accuracy:
85.25%



#### Comments:
- The accuracy has improved from 80.33% to 85.25%

In [34]:
param_grid_DT = {
    'criterion': ['entropy'],
    'max_depth': [3,4,5,6,7],
    'min_samples_split': [8,9,10,11,12],
    'min_samples_leaf': [3,4,5],
    'max_features': ['log2']
}

GSCV_DT_model = GridSearchCV(
    estimator=DT,
    param_grid=param_grid_DT,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

GSCV_DT_model.fit(X_train, y_train)
y_pred_GSCV_DT = GSCV_DT_model.predict(X_test)


print("\nFinal best parameters found by GridSearchCV:")
print(GSCV_DT_model.best_params_)
print(f"Accuracy of Grid Search Model: {accuracy_score(y_test, y_pred_GSCV_DT) * 100:.2f}%")

final_DT_model = RSCV_DT_model.best_estimator_


report_DT = classification_report(y_test, y_pred_RSCV_DT, output_dict=True)
accuracy_DT = report_DT['accuracy']
precision_DT = report_DT['weighted avg']['precision']
recall_DT = report_DT['weighted avg']['recall']
f1_score_DT = report_DT['weighted avg']['f1-score']


comparison_df.loc["Decision Tree Tuned"] = [accuracy_DT, precision_DT, recall_DT, f1_score_DT]

print(f"\n Final Model Accuracy: {accuracy_DT * 100:.2f}%")
print("\nClassification Report:")
print(pd.DataFrame(report_DT))
print(comparison_df)

Fitting 5 folds for each of 75 candidates, totalling 375 fits

Final best parameters found by GridSearchCV:
{'criterion': 'entropy', 'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 3, 'min_samples_split': 11}
Accuracy of Grid Search Model: 77.05%

 Final Model Accuracy: 85.25%

Classification Report:
                   0          1  accuracy  macro avg  weighted avg
precision   0.794118   0.925926  0.852459   0.860022      0.863263
recall      0.931034   0.781250  0.852459   0.856142      0.852459
f1-score    0.857143   0.847458  0.852459   0.852300      0.852062
support    29.000000  32.000000  0.852459  61.000000     61.000000
                            accuracy  precision    recall  f1_score
Logistic Regression         0.885246   0.885477  0.885246  0.885122
Decision Tree               0.803279   0.809808  0.803279  0.803067
Random Forest               0.836066   0.842949  0.836066  0.835889
Support Vector Machine SVM  0.885246   0.885845  0.885246  0.885308
Logistic Re

#### Comments:
- Grid Search has lower Accuracy than Random Search as Grid Search chose the model with the highest average score during cross-validation on the training data.
- I will use the Random Search model as it generalizes much better

### 3- Random Forest 

In [35]:
from sklearn.ensemble import RandomForestClassifier

RF = RandomForestClassifier(random_state= 42)

param_dist_RF = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [5, 10, 15, 20, 25, 30, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', None]
}

RSCV_RF_model = RandomizedSearchCV(
    estimator=RF,
    param_distributions=param_dist_RF,
    n_iter=150,
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

RSCV_RF_model.fit(X_train, y_train)

y_pred_RSCV_RF = RSCV_RF_model.predict(X_test)


print("\nFinal best parameters found by GridSearchCV:")
print(RSCV_RF_model.best_params_)
print(f"Accuracy: {accuracy_score(y_test, y_pred_RSCV_RF) * 100:.2f}%")

Fitting 5 folds for each of 150 candidates, totalling 750 fits


KeyboardInterrupt: 

#### Comment: Accuracy increased from  83.60% to 88.52% after using Random Search

In [ ]:
param_grid_RF = {
    'n_estimators': [180,190,200,210,220],
    'max_depth': [28,29,30,31,32],
    'min_samples_split': [13,14,15,16,17],
    'min_samples_leaf': [4,5,6,7,8],
    'max_features': [None]
}


GSCV_RF_model = GridSearchCV(
    estimator=RF,
    param_grid=param_grid_RF,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

GSCV_RF_model.fit(X_train, y_train)
y_pred_GSCV_RF = GSCV_RF_model.predict(X_test)


print("\nFinal best parameters found by GridSearchCV:")
print(GSCV_RF_model.best_params_)

final_RF_model = GSCV_RF_model.best_estimator_


report_RF = classification_report(y_test, y_pred_GSCV_RF, output_dict=True)
accuracy_RF = report_RF['accuracy']
precision_RF = report_RF['weighted avg']['precision']
recall_RF = report_RF['weighted avg']['recall']
f1_score_RF = report_RF['weighted avg']['f1-score']


comparison_df.loc["Random Forest Tuned"] = [accuracy_RF, precision_RF, recall_RF, f1_score_RF]

print(f"\n Final Model Accuracy: {accuracy_RF * 100:.2f}%")
print("\nClassification Report:")
print(pd.DataFrame(report_RF))
print(comparison_df)

Fitting 5 folds for each of 625 candidates, totalling 3125 fits

Final best parameters found by GridSearchCV:
{'max_depth': 28, 'max_features': None, 'min_samples_leaf': 8, 'min_samples_split': 13, 'n_estimators': 220}

 Final Model Accuracy: 88.52%

Classification Report:
                   0          1  accuracy  macro avg  weighted avg
precision   0.843750   0.931034  0.885246   0.887392      0.889539
recall      0.931034   0.843750  0.885246   0.887392      0.885246
f1-score    0.885246   0.885246  0.885246   0.885246      0.885246
support    29.000000  32.000000  0.885246  61.000000     61.000000
                            accuracy  precision    recall  f1_score
Logistic Regression         0.885246   0.885477  0.885246  0.885122
Decision Tree               0.803279   0.809808  0.803279  0.803067
Random Forest               0.836066   0.842949  0.836066  0.835889
Support Vector Machine SVM  0.885246   0.885845  0.885246  0.885308
Logistic Regression Tuned   0.901639   0.901639  0.

#### Comment: Same accuracy as Random Search but the parameters has been refined

### 4- Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC

svm = SVC(random_state=42)

param_dist_SVM = {
     'C': [0.1, 1, 10, 100, 1000],
    'gamma': [1, 0.1, 0.01, 0.001, 0.0001, 'scale'],
    'kernel': ['rbf', 'linear']
}

RSCV_SVM_model = RandomizedSearchCV(
    estimator=svm,
    param_distributions=param_dist_SVM,
    n_iter=150,
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

RSCV_SVM_model.fit(X_train, y_train)

y_pred_RSCV_SVM = RSCV_SVM_model.predict(X_test)


print("\nFinal best parameters found by GridSearchCV:")
print(RSCV_SVM_model.best_params_)
print(f"Accuracy: {accuracy_score(y_test, y_pred_RSCV_SVM) * 100:.2f}%")

d:\Heart_Disease_Project\venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 60 is smaller than n_iter=150. Running 60 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 60 candidates, totalling 300 fits

Final best parameters found by GridSearchCV:
{'kernel': 'rbf', 'gamma': 0.1, 'C': 10}
Accuracy: 83.61%


#### Comment: It has a lower Accuracy than using the default hyperparameters which got 88.52%

In [ ]:
param_grid_SVM = {
    'C':      [1, 5, 10, 15, 20], 
    'gamma':  [0.05, 0.1, 0.5, 1, 'scale'],
    'kernel': ['rbf']
}


GSCV_SVM_model = GridSearchCV(
    estimator=svm,
    param_grid=param_grid_SVM,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

GSCV_SVM_model.fit(X_train, y_train)
y_pred_GSCV_SVM = GSCV_SVM_model.predict(X_test)


print("\nFinal best parameters found by GridSearchCV:")
print(GSCV_SVM_model.best_params_)
print(f"Accuracy: {accuracy_score(y_test, y_pred_GSCV_SVM) * 100:.2f}%")

final_SVM_model = RSCV_SVM_model.best_estimator_


report_SVM = classification_report(y_test, y_pred_GSCV_SVM, output_dict=True)
accuracy_SVM = report_SVM['accuracy']
precision_SVM = report_SVM['weighted avg']['precision']
recall_SVM = report_SVM['weighted avg']['recall']
f1_score_SVM = report_SVM['weighted avg']['f1-score']


comparison_df.loc["Support Vector Machine SVM Tuned"] = [accuracy_SVM, precision_SVM, recall_SVM, f1_score_SVM]

print(f"\n Final Model Accuracy: {accuracy_SVM * 100:.2f}%")
print("\nClassification Report:")
print(pd.DataFrame(report_SVM))
print(comparison_df)

Fitting 5 folds for each of 25 candidates, totalling 125 fits

Final best parameters found by GridSearchCV:
{'C': 15, 'gamma': 0.1, 'kernel': 'rbf'}
Accuracy: 83.61%

 Final Model Accuracy: 83.61%

Classification Report:
                   0          1  accuracy  macro avg  weighted avg
precision   0.787879   0.892857  0.836066   0.840368      0.842949
recall      0.896552   0.781250  0.836066   0.838901      0.836066
f1-score    0.838710   0.833333  0.836066   0.836022      0.835889
support    29.000000  32.000000  0.836066  61.000000     61.000000
                                  accuracy  precision    recall  f1_score
Decision Tree                     0.803279   0.809808  0.803279  0.803067
Decision Tree Tuned               0.852459   0.863263  0.852459  0.852062
Logistic Regression               0.885246   0.885477  0.885246  0.885122
Logistic Regression Tuned         0.901639   0.901639  0.901639  0.901639
Random Forest                     0.836066   0.842949  0.836066  0.835889


#### Comment: It has the same accuracy as RandomSearchCV, I will choose the model by GridSearchCV

## Compare optimized models with baseline performance.

In [ ]:
comparison_df = comparison_df.sort_index(ascending=True)
print(comparison_df)

output_csv_path = 'D:\\Heart_Disease_Project\\results\\evaluation_metrics.csv'
comparison_df.to_csv(output_csv_path, index=True)

                                  accuracy  precision    recall  f1_score
Decision Tree                     0.803279   0.809808  0.803279  0.803067
Decision Tree Tuned               0.852459   0.863263  0.852459  0.852062
Logistic Regression               0.885246   0.885477  0.885246  0.885122
Logistic Regression Tuned         0.901639   0.901639  0.901639  0.901639
Random Forest                     0.836066   0.842949  0.836066  0.835889
Random Forest Tuned               0.836066   0.842949  0.836066  0.835889
Support Vector Machine SVM        0.885246   0.885845  0.885246  0.885308
Support Vector Machine SVM Tuned  0.836066   0.842949  0.836066  0.835889


#### Comments: 
- Decision Tree has improved after tuning
- Logistic Regression has improved after tuning
- Random Forest has the same metrics as without tuning which means it has already reached its best results using the default hyperparameters
- Support Vector Machine metrics has degraded after tuning as the tuned model likely overfitted on the cross-validation folds, leading to worse generalization.

## Final Result: I will use Logistic Regression Tuned Model as it has the best metric of them all

## Creating the final model's pipeline

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel




X_Raw = pd.read_csv('D:\\Heart_Disease_Project\\data\\heart_disease_features_Raw.csv')
y_Raw = pd.read_csv('D:\\Heart_Disease_Project\\data\\heart_disease_target_Raw.csv').squeeze()

y_Raw = (y_Raw > 0).astype(int)  

X_train_Raw, X_test_Raw, y_train_Raw, y_test_Raw = train_test_split(
    X_Raw, y_Raw, test_size=0.2, random_state=42
)


class TargetBinarizer(BaseEstimator, TransformerMixin):
    def fit(self, y, *_):
        return self
    def transform(self, y):
        return (y > 0).astype(int)


class RoundCategoricalInts(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns if len(self.columns) == X.shape[1] else None)
        for col in self.columns:
            if col in X.columns:
                X[col] = X[col].round().astype(int)
        return X.values


class OutlierCapper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = pd.DataFrame(X)
        for col in X.columns:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            X[col] = np.clip(X[col], lower, upper)
        return X.values



categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']

special_int_cols_cat = ['thal']  
special_int_cols_num = ['ca']    



numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rounder', RoundCategoricalInts(columns=special_int_cols_num)), 
    ('outlier_capper', OutlierCapper()),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('rounder', RoundCategoricalInts(columns=special_int_cols_cat)), 
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

pipeline_model = LogisticRegression(
    C=0.2, l1_ratio=0.25, penalty='elasticnet',
    solver='saga', max_iter=5000, random_state=42
)


feature_selector = SelectFromModel(pipeline_model)



pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', feature_selector),
    ('classifier', pipeline_model)
])

## Saving the final model's pipeline

In [ ]:
import joblib

joblib.dump(pipeline, 'D:\\Heart_Disease_Project\\models\\model_pipeline.pkl')

['D:\\Heart_Disease_Project\\models\\model_pipeline.pkl']

## Loading it and evaluating it

In [38]:
pipeline = joblib.load('D:\\Heart_Disease_Project\\models\\model_pipeline.pkl')


pipeline.fit(X_train_Raw, y_train_Raw)

y_pred = pipeline.predict(X_test_Raw)

print(classification_report(y_test_Raw,y_pred))

              precision    recall  f1-score   support

           0       0.87      0.90      0.88        29
           1       0.90      0.88      0.89        32

    accuracy                           0.89        61
   macro avg       0.88      0.89      0.89        61
weighted avg       0.89      0.89      0.89        61

